# Lesson 18: Human-in-the-Loop — Approval Before Multiply

## Objective
Pause graph execution **before** a multiply operation, display the pending computation to a human for approval, then **resume** (or reject) based on their input.

## Limitation of Lesson 17
- ❌ The graph ran to completion autonomously — no human review possible
- ❌ Dangerous operations (large numbers, important decisions) proceeded without checks

## What is NEW in Lesson 18?
- ✅ `interrupt_before=["node_name"]` — pause execution BEFORE a node runs
- ✅ `interrupt_after=["node_name"]` — pause execution AFTER a node runs
- ✅ `graph.invoke(None, config=config)` — resume a paused graph
- ✅ `graph.update_state(config, values)` — inject new values before resuming
- ✅ Human-in-the-loop (HITL) design pattern

## Theory: Interrupt & Resume

```
invoke(state) → runs until interrupt point → PAUSES
                                               ↓
                                         Human reviews
                                               ↓
                  invoke(None, config) ← Human approves
                  (resume from pause point)
```

When `interrupt_before=["multiply"]`:
1. Graph runs normally until it's about to run `multiply`
2. Graph PAUSES — `invoke()` returns current state
3. Human inspects the pending operation
4. `invoke(None, config)` RESUMES from the pause point
5. `multiply` node runs and graph completes

⚠️ Requires a **checkpointer** — the paused state is saved in the checkpoint.


In [ ]:
# ── Cell 1: Imports + LLM Setup ──────────────────────────────────────────────
from dotenv import load_dotenv
import warnings
warnings.filterwarnings("ignore")

import os
import vertexai
from langchain_google_vertexai import ChatVertexAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import interrupt, Command
from typing import TypedDict, Optional, Literal, Annotated
import operator
from IPython.display import Image, display

load_dotenv()
vertexai.init(project=os.getenv("PROJECT_ID"), location=os.getenv("LOCATION"))
llm = ChatVertexAI(model="gemini-2.5-flash", temperature=0)

print("✓ Setup complete")
print("  interrupt: LangGraph primitive for pausing execution")
print("  Command:   used to resume with updated state")


In [ ]:
# ── Cell 2: State Schema ────────────────────────────────────────────────────
class State(TypedDict):
    a: float
    b: float
    operation: str
    result: Optional[float]
    approved: bool         # Did human approve?
    human_feedback: str    # Human's response
    history: Annotated[list, operator.add]

print("✓ State with approval fields defined")


In [ ]:
# ── Cell 3: Parse Node (uses LLM) ────────────────────────────────────────────
PARSE_PROMPT = "Parse the math question. Respond EXACTLY:\nOPERATION: <add|multiply|divide>\nA: <number>\nB: <number>"

def parse_node(state: State) -> dict:
    print(f"  [parse] Parsing operation='{state['operation']}'")
    return {}  # Operation already set in this demo

def add_node(state: State) -> dict:
    res = state["a"] + state["b"]
    entry = f"{state['a']} + {state['b']} = {res}"
    print(f"  [add] {entry}")
    return {"result": res, "history": [entry]}

def divide_node(state: State) -> dict:
    if state["b"] == 0:
        return {"result": None, "history": ["divide: b=0 error"]}
    res = state["a"] / state["b"]
    entry = f"{state['a']} ÷ {state['b']} = {res:.4f}"
    print(f"  [divide] {entry}")
    return {"result": res, "history": [entry]}

print("✓ add_node and divide_node defined (no approval needed)")


In [ ]:
# ── Cell 4: Multiply Node — Requires Approval ────────────────────────────────
# This node will be INTERRUPTED before it runs
# The interrupt() call INSIDE the node pauses execution and waits for human input

def multiply_node(state: State) -> dict:
    a, b = state["a"], state["b"]
    
    # ── INTERRUPT: pause and ask for human approval ──────────────────────────
    # interrupt() suspends execution here and returns the value to the caller
    # When resumed, execution continues from this exact line
    human_decision = interrupt({
        "question": f"⚠️  About to multiply {a} × {b} = {a*b}. Approve?",
        "pending_operation": f"{a} × {b}",
        "expected_result": a * b
    })
    # ── Execution resumes here after human responds ──────────────────────────
    
    if human_decision.get("approved", False):
        res = a * b
        entry = f"{a} × {b} = {res} [APPROVED by human]"
        print(f"  [multiply] ✓ Approved! {entry}")
        return {"result": res, "approved": True, "history": [entry]}
    else:
        entry = f"{a} × {b} = REJECTED by human"
        print(f"  [multiply] ✗ Rejected! {entry}")
        return {"result": None, "approved": False, 
                "human_feedback": human_decision.get("reason", ""),
                "history": [entry]}

print("✓ multiply_node with interrupt() defined")
print("  interrupt() will PAUSE execution and wait for human response")


In [ ]:
# ── Cell 5: Router ──────────────────────────────────────────────────────────
def router(state: State) -> Literal["add", "multiply", "divide"]:
    return state["operation"]

print("✓ Router defined")


In [ ]:
# ── Cell 6: Build Graph with Interrupt ───────────────────────────────────────
memory = MemorySaver()   # Required for interrupt/resume

builder = StateGraph(State)
builder.add_node("add",      add_node)
builder.add_node("multiply", multiply_node)
builder.add_node("divide",   divide_node)

builder.add_conditional_edges(START, router, {
    "add":      "add",
    "multiply": "multiply",
    "divide":   "divide",
})
builder.add_edge("add",      END)
builder.add_edge("multiply", END)
builder.add_edge("divide",   END)

# ── Compile with checkpointer (REQUIRED for interrupt/resume) ────────────────
graph = builder.compile(checkpointer=memory)

print("✓ Graph compiled with MemorySaver (required for interrupt/resume)")


In [ ]:
# ── Cell 7: Visualize ───────────────────────────────────────────────────────
display(Image(graph.get_graph().draw_mermaid_png()))


In [ ]:
# ── Cell 8: Test Add (No Interrupt) ──────────────────────────────────────────
print("="*55)
print("Test: operation='add' — no interrupt needed")
print("="*55)

config_add = {"configurable": {"thread_id": "add-test-1"}}
state = {"a": 12.0, "b": 8.0, "operation": "add",
         "result": None, "approved": False, "human_feedback": "", "history": []}

out = graph.invoke(state, config=config_add)
print(f"\nResult: {out['result']}  (completed without interrupt)")


In [ ]:
# ── Cell 9: Test Multiply — PAUSE and APPROVE ────────────────────────────────
print("="*55)
print("Test: operation='multiply' — INTERRUPT occurs!")
print("="*55)

config_mul = {"configurable": {"thread_id": "multiply-approval-1"}}
state = {"a": 7.0, "b": 8.0, "operation": "multiply",
         "result": None, "approved": False, "human_feedback": "", "history": []}

print("\nStep 1: invoke() — graph runs until multiply node...")
result = graph.invoke(state, config=config_mul)
print(f"\nGraph PAUSED! invoke() returned: {result}")
print("  ↑ Notice: result is still None — multiply hasn't run yet")

# Inspect paused state
paused = graph.get_state(config_mul)
print(f"\nPaused state:")
print(f"  Next node to run: {paused.next}")
print(f"  a={paused.values['a']}, b={paused.values['b']}")

# Check the interrupt value
if hasattr(paused, 'tasks') and paused.tasks:
    for task in paused.tasks:
        if hasattr(task, 'interrupts') and task.interrupts:
            for intr in task.interrupts:
                print(f"\n  Interrupt message: {intr.value}")


In [ ]:
# ── Cell 10: Human Reviews and APPROVES ──────────────────────────────────────
print("\nStep 2: Human approves the multiplication...")

# Resume by invoking with Command(resume={...})
from langgraph.types import Command

out = graph.invoke(
    Command(resume={"approved": True}),   # Human's approval
    config=config_mul
)

print(f"\nGraph RESUMED and completed!")
print(f"  Result: {out['result']}")
print(f"  Approved: {out['approved']}")
print(f"  History: {out['history']}")


In [ ]:
# ── Cell 11: Test Multiply — PAUSE and REJECT ────────────────────────────────
print("="*55)
print("Test: Multiply with REJECTION")
print("="*55)

config_rej = {"configurable": {"thread_id": "multiply-reject-1"}}
state2 = {"a": 999.0, "b": 999.0, "operation": "multiply",
          "result": None, "approved": False, "human_feedback": "", "history": []}

# Run to interrupt
graph.invoke(state2, config=config_rej)
print("Graph paused — human sees: 999 × 999 = 998001")
print("Human decides to REJECT...")

# Reject
out_rej = graph.invoke(
    Command(resume={"approved": False, "reason": "Too large a number"}),
    config=config_rej
)
print(f"\nResult: {out_rej['result']}  (None = rejected)")
print(f"Human feedback: {out_rej['human_feedback']}")
print(f"History: {out_rej['history']}")


## Interrupt Patterns

### Pattern 1: `interrupt()` inside a node (this lesson)
```python
def my_node(state):
    decision = interrupt({"question": "Approve?"})
    # Execution pauses here
    # When resumed with Command(resume=value):
    # decision = value
    if decision["approved"]: ...
```

### Pattern 2: `interrupt_before` at compile time
```python
graph = builder.compile(
    checkpointer=memory,
    interrupt_before=["multiply"]   # Pause BEFORE multiply runs
)
```

### Resume
```python
# Resume with new input
graph.invoke(Command(resume=value), config=config)

# Or resume without new input (use state as-is)
graph.invoke(None, config=config)
```

## Production Use Cases for HITL
- ✅ Approve before sending emails / making payments
- ✅ Review before deploying code
- ✅ Validate LLM-generated SQL before execution
- ✅ Flag uncertain predictions for human review
- ✅ Compliance checkpoints in financial workflows

## Summary

| | Autonomous (Lesson 17) | HITL (Lesson 18) |
|--|----------------------|----------------|
| Execution | Runs to END | Pauses at interrupt |
| Human role | None | Reviews + approves |
| Resume | N/A | `invoke(None)` or `Command(resume=...)` |

## Limitations → What Lesson 19 Solves
- ❌ What if we want to go BACK to an earlier step and replay from there?
- ❌ What if we want to see what would have happened with different inputs?
- ❌ **Lesson 19**: Time travel — replay from any checkpoint with modified state
